[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/boltzgen/advanced/06_advanced_ai/06_advanced_filtering.ipynb)


# 06 — 고급 필터링·자동화

> 본문 [`06_advanced_ai.md`](06_advanced_ai.md) 와 **한 절씩 짝지어** 보세요.
> **앞 랩에서 이어져요** — Ch.05 · Ch.04 의 `my_run/` 을 먼저 찾고, 없으면 커밋된 `data/` 로 대신합니다.


## 0) 준비 — 저장소 클론 & 작업 폴더 이동

이 셀이 저장소를 클론하고 `06_advanced_ai/` 로 이동합니다. 로컬에서 `06_advanced_ai/` 안에 열었다면 클론 없이 진행돼요.

In [ ]:
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # fork 했다면 본인 주소로 바꾸세요
CLONE_AS = "bio-guides"
CHAPTER  = "06_advanced_ai"
PIP_PKGS = "pandas matplotlib"   # 없으면 설치할 분석 라이브러리

import os, sys, subprocess, pathlib
IN_COLAB = "google.colab" in sys.modules

# HF 가중치 다운로드가 멈춘 채 매달리지 않도록 타임아웃을 겁니다.
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")   # 스트림 30초 무응답 → 끊고 재시도
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "15")

def _run(cmd, quiet=False):
    """quiet=True 면 출력을 삼키고 **실패했을 때만** 보여줘요.
    apt-get 은 "(Reading database ... 5%(Reading database ... 10%" 같은 진행률을 600자 넘게 쏟아내는데,
    그게 노트북을 연 학습자가 보는 첫 화면을 덮어버려서 실패로 오해하게 만들거든요."""
    print("$", cmd)
    if not quiet:
        subprocess.run(cmd, shell=True, check=True); return
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.returncode != 0:
        print((p.stdout or "") + (p.stderr or ""))
        raise subprocess.CalledProcessError(p.returncode, cmd)

_MARK = "boltzgen_viz.py"          # 이 파일이 있는 폴더가 advanced/ 루트

def _find_root():
    """advanced/ 루트를 찾습니다."""
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):   # 클론 직후: cwd 아래로만 깊이 3까지
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "advanced/ 루트를 못 찾았습니다. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

ADV_ROOT = ROOT.resolve()
os.chdir(ADV_ROOT / CHAPTER)          # 이 챕터 폴더로 이동 → data/ 상대경로가 그대로 동작
sys.path.insert(0, str(ADV_ROOT))     # boltzgen_viz import 보장
import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    # Colab 에는 한글 폰트가 없어 그래프의 한글 라벨이 □ 로 깨집니다.
    _run("apt-get -qq update", quiet=True)
    _run("DEBIAN_FRONTEND=noninteractive apt-get -qq install -y fonts-nanum", quiet=True)

# import 안 되는 패키지만 설치합니다.
import importlib, importlib.util
_IMPORT_NAME = {"scikit-learn": "sklearn", "pillow": "PIL", "biopython": "Bio"}
def _have(mod):
    try: return importlib.util.find_spec(mod) is not None
    except Exception: return False
def _ensure(pkgs):
    miss = [p for p in pkgs.split() if not _have(_IMPORT_NAME.get(p, p))]
    if miss:
        print("필요 라이브러리 설치:", " ".join(miss))
        _run(f'"{sys.executable}" -m pip -q install ' + " ".join(miss))  # python -m pip (Ch.03 권고)
        importlib.invalidate_caches()
if PIP_PKGS:
    _ensure(PIP_PKGS)

# 내 결과는 my_run/ 에 쌓이고, 없으면 커밋된 레퍼런스로 폴백합니다.
MY  = pathlib.Path("my_run")
MY.mkdir(exist_ok=True)

def find_one(pattern, ref, quiet=False):
    """my_run/ 에서 먼저 찾고, 없으면 레퍼런스 폴더에서 찾습니다."""
    for base in (MY/"final_ranked_designs", MY/"intermediate_designs_inverse_folded", MY):
        hits = sorted(base.glob(pattern))
        if hits:
            if not quiet: print(f"[내 결과]   {hits[0]}")
            return hits[0]
    hits = sorted(pathlib.Path(ref).glob(pattern))
    assert hits, f"{ref} 에서 '{pattern}' 을 찾지 못했습니다."
    if not quiet: print(f"[레퍼런스] {hits[0]}")
    return hits[0]

def cols_in(df, *names):
    """내 실행 결과와 레퍼런스는 컬럼 구성이 조금 다를 수 있어, 있는 컬럼만 고른다."""
    missing = [c for c in names if c not in df.columns]
    if missing:
        print("(이 실행에는 없는 컬럼 — 건너뜁니다:", ", ".join(missing) + ")")
    return [c for c in names if c in df.columns]

def inherit_run(*rel_paths):
    """앞 챕터에서 돌린 설계 결과를 이어받습니다(없으면 레퍼런스로 폴백)."""
    global MY
    if (MY / "final_ranked_designs").exists():
        return MY
    for rel in rel_paths:
        p = pathlib.Path(rel)
        if (p / "final_ranked_designs").exists():
            print("[이어받기] 앞 챕터에서 직접 돌린 결과를 사용합니다:", p)
            MY = p
            return MY
    return MY

# 레퍼런스 그림을 덮어쓰지 않도록 my_ 접두어.
def my_fig(name):
    return f"my_{name}"

print("작업 폴더 :", pathlib.Path.cwd())

## 1) 선별 전 전체를 펼치기 (본문 6.2)

Ch.05 에서 본 최종 10개는 여기 있는 디자인들에서 걸러진 결과예요. 생성(design~analysis)은 무겁고
선별(filtering)은 몇 초면 끝나니, 한 번 만든 `all_designs_metrics.csv` 위에서 기준만 바꿔가며 실험합니다.

아래 표는 앞부분 미리보기가 아니라 전체 행이에요 — Colab 은 표 아래 페이지 컨트롤로 넘겨 보고,
주피터는 앞뒤만 잘라 `N rows × M columns` 로 표시합니다.

In [ ]:
import pandas as pd
# Ch.05 또는 Ch.04 에서 직접 돌린 결과가 있으면 그걸 이어서 씁니다(먼저 찾는 쪽 우선)
inherit_run("../05_result_interpretation/my_run", "../04_basic_usage/my_run")
df = pd.read_csv(find_one("all_designs_metrics.csv", "data/vanilla"))
print("선별 전 디자인", len(df), "개 | 컬럼", df.shape[1], "종")
if "pass_filters" in df.columns:
    print("내장 필터를 전부 통과한 디자인", int(df["pass_filters"].astype(bool).sum()), "개")
df[cols_in(df, "id", "design_ptm", "design_to_target_iptm", "filter_rmsd",
           "plip_hbonds_refolded", "num_filters_passed", "pass_filters")]

## 2) 부등호는 통과 조건이다 (본문 6.4)

방금 표의 `pass_*` 는 BoltzGen 이 스스로 매긴 판정이에요. 레퍼런스 100개에서는 이 판정을 전부 통과한 게
3개뿐인데, 그 셋이 Ch.05 최종 순위의 1~3위였어요. 이제 그 판정을 내가 직접 쓰려면
`--additional_filters 'feature<threshold'` 를 쓰는데, 여기서 부등호를 거꾸로 읽는 실수가 잦아요.

- `<` 는 이 값 이하만 통과 (작을수록 좋은 지표)
- `>` 는 이 값 이상만 통과 (클수록 좋은 지표)

그래서 Ala 가 너무 많은 디자인을 버리고 싶으면 `design_ALA<0.3` 이 맞아요. `design_ALA>0.3` 이라고 쓰면
정반대로 Ala 가 30% 넘는 것만 남습니다. 본문의 `design_ALA` 는 CSV 에서 `ALA_fraction` 컬럼으로 나오니,
내장 판정 `pass_ALA_fraction_filter` 와 직접 계산한 `ALA_fraction < 0.3` 이 같은지부터 맞춰 보죠.

In [ ]:
ALA_T = 0.3                                   # 내장 Ala-rich 배제 임계값
keep = df["ALA_fraction"] < ALA_T              # 'design_ALA<0.3' 이 남기는 쪽
drop = ~keep
print(f"전체 {len(df)}개 | 'ALA_fraction<0.3' 통과 {int(keep.sum())}개 | 탈락(Ala-rich) {int(drop.sum())}개")
if "pass_ALA_fraction_filter" in df.columns:
    agree = int((keep == df["pass_ALA_fraction_filter"].astype(bool)).sum())
    print(f"내장 pass_ALA_fraction_filter 와 일치 {agree}/{len(df)}")

flip = df.loc[drop, cols_in(df, "id", "ALA_fraction", "design_ptm", "design_to_target_iptm")] \
         .sort_values("ALA_fraction", ascending=False)
print(f"\n부등호를 'ALA_fraction>0.3' 으로 뒤집으면 남는 건 이 {len(flip)}개 — 버리려던 것만 정확히 남아요")
print(flip.to_string(index=False))

## 3) `--filter_biased` — 조성 이상치를 기본으로 거른다 (본문 6.4)

방금 본 Ala-rich 배제는 사실 기본으로 켜져 있어요(`--filter_biased`, 기본 true).
생성 모델은 종종 ALA·GLY·GLU·LEU·VAL 을 몰아 넣은 "치트성" 서열을 만드는데, 이 다섯 잔기의 비율을
각각 보고 넘치면 탈락 표시를 남깁니다. 정말 버릴 만한 것들인지 pTM 으로 확인해 보죠.

In [ ]:
comp = cols_in(df, "pass_ALA_fraction_filter", "pass_GLY_fraction_filter",
               "pass_GLU_fraction_filter", "pass_LEU_fraction_filter", "pass_VAL_fraction_filter")
comp_ok = df[comp].astype(bool).all(axis=1)
print(f"조성 필터 {len(comp)}종을 모두 통과 {int(comp_ok.sum())}/{len(df)}")
for name in comp:
    print(f"  {name:34s} 탈락 {int((~df[name].astype(bool)).sum()):3d}개")

rich = df["ALA_fraction"] >= ALA_T
print(f"\nAla-rich {int(rich.sum())}개 평균 pTM {df.loc[rich, 'design_ptm'].mean():.3f}"
      f" | 나머지 {int((~rich).sum())}개 평균 pTM {df.loc[~rich, 'design_ptm'].mean():.3f}")

## 4) 메트릭 필터 위에 조성 필터를 겹치기 (본문 6.4)

레퍼런스 100개에서는 Ala-rich 24개의 평균 pTM 이 **0.381**, 나머지 76개가 **0.660** 이었어요.
조성만 보고도 걸러지는 게 이만큼 있다는 뜻이니, 결합·구조 메트릭으로 먼저 좁힌 뒤 조성 필터를 얹어 봅니다.

In [ ]:
m = (df["design_to_target_iptm"] > 0.5) & (df["design_ptm"] > 0.7) & (df["filter_rmsd"] < 2.0)
mc = m & comp_ok
print(f"메트릭 필터만 (ipTM>0.5, pTM>0.7, RMSD<2.0) {int(m.sum())}/{len(df)}")
print(f"조성 필터까지 겹치면 {int(mc.sum())}/{len(df)} — 조성 때문에 추가 탈락 {int((m & ~mc).sum())}개")

sel = df.loc[m, cols_in(df, "id", "design_to_target_iptm", "design_ptm", "filter_rmsd",
                        "ALA_fraction", "GLY_fraction", "num_design")].copy()
sel["comp_pass"] = comp_ok[m].values
sel.sort_values("design_to_target_iptm", ascending=False)

In [ ]:
from boltzgen_viz import load_metrics, compare_bars
rows = load_metrics(str(find_one("all_designs_metrics.csv", "data/vanilla", quiet=True)))
g_metric = [r for r in rows if r["design_to_target_iptm"] > 0.5
            and r["design_ptm"] > 0.7 and r["filter_rmsd"] < 2.0]
g_both = [r for r in g_metric if float(r.get("ALA_fraction", 0)) < ALA_T]
FIG = my_fig("06_filter_compare.png")
compare_bars({"all designs": rows, "+ metric": g_metric, "+ composition": g_both},
             "design_to_target_iptm", "Filtering Effect — mean ipTM",
             "mean ipTM", FIG, thr=0.5, thr_label="Good (0.5)")
from IPython.display import Image; Image(FIG)

## 5) 남은 세 노브 — 크기·다양성·가중치 (본문 6.4)

필터를 통과한 것만 모으면 길이가 한쪽으로 쏠릴 수 있어요. `--size_buckets` 는 길이 구간마다 최대 몇 개를
뽑을지 못 박아 다양한 크기를 확보합니다. 방금 통과한 디자인들이 실제로 쏠렸는지부터 보죠.

In [ ]:
for lo, hi in [(80, 100), (100, 140)]:
    inb = (df["num_design"] >= lo) & (df["num_design"] < hi)
    print(f"길이 {lo}-{hi} : 전체 {int(inb.sum()):3d}개 중 메트릭 통과 {int((inb & m).sum())}개")
print("\n한 구간에 쏠렸다면 --size_buckets 로 구간별 상한을 걸어 반대쪽 크기도 남깁니다.")

나머지 둘(`--metrics_override`·`--alpha`)은 걸러내는 게 아니라 순위 자체를 바꿔요. 네 노브를 한 번에 정리하면 이렇습니다.

| 옵션 | 의미 |
|------|------|
| `--metrics_override k=w` | 메트릭별 가중치. 값이 클수록 그 메트릭을 덜 중요하게(down-weight) 봅니다 |
| `--alpha 0~1` | 0 이면 품질만, 1 이면 다양성만. 비슷한 서열이 상위를 덮을 때 올립니다 |
| `--size_buckets 80-100:5` | 길이 구간별 선발 상한 |
| `--filter_biased true` | 조성 이상치 배제(기본값 true) |

지금까지 pandas 로 실험한 걸 그대로 명령으로 옮기면 이렇게 돼요.

In [ ]:
cmd = [
    "boltzgen run spec.yaml --output out --steps filtering",
    "  --metrics_override plip_hbonds_refolded=4 delta_sasa_refolded=2",
    "  --additional_filters 'design_ALA<0.3' 'design_GLY<0.2' 'designfolding-filter_rmsd<2.0'",
    "  --size_buckets 80-100:5 100-140:5",
    "  --alpha 0.3 --filter_biased true",
]
print(" \\\n".join(cmd))
print("\n생성(design~analysis)은 한 번, filtering 은 기준을 바꿔가며 여러 번 — 이게 본문 6.2 의 핵심 패턴이에요.")

## 6) 계층적 스크리닝 — 한 판이 아니라 라운드로 (본문 6.1)

지금까지 한 건 "한 번 만든 결과를 여러 기준으로 다시 걸러 보기"였어요. 실전에서는 이 걸러내기를 라운드로 쌓습니다.

```
Level 1  넓고 얕게 — boltzgen run spec.yaml --num_designs 10000 --budget 200 --output L1
             ↓ 상위 후보에서 어느 결합 전략이 되는지 파악 → YAML 제약 보강(결합부위 좁히기 등)
Level 2  좁고 깊게 — boltzgen run spec_refined.yaml --num_designs 5000 --budget 20 --output L2
             ↓
Level 3  최종 검증 — 실험 가능한 top 5~10
```

핵심은 Level 이 올라갈수록 `num_designs` 는 줄이고 제약은 강화한다는 거예요. Level 1 에서 방향을 잡고
Level 2 에서 그 방향만 집중 탐색하니, 비용은 줄고 품질은 올라갑니다.

## 7) 라운드를 손으로 돌리지 않기 — 스윕과 merge (본문 6.3)

Level 1 의 운영점(`num_designs` × `budget`)을 감으로 정하지 말고 스윕으로 찾습니다.
GPU 가 하나면 동시 실행은 메모리 경합·OOM 을 부르니 순차로 돌리고, 큰 작업은 `--num_designs` 를 쪼개
여러 번 돌린 뒤 `boltzgen merge` 로 합치는 게 정석이에요(메모리 안정성 + 중단 복구).

In [ ]:
import itertools
outs = []
for budget, num in itertools.product([20, 50, 100], [1000, 5000]):
    out = f"sweep/b{budget}_n{num}"
    outs.append(out)
    print(f"boltzgen run design.yaml --protocol protein-anything --output {out} "
          f"--num_designs {num} --budget {budget}")
print("\nboltzgen merge " + " ".join(outs) + " --output sweep/merged")
print("\n각 실행의 final_ranked_designs/final_designs_metrics_<budget>.csv 를 pandas 로 모아")
print("groupby(['num_designs','budget'])['design_to_target_iptm'].mean() 으로 운영점을 고릅니다.")

## 8) 커스텀 scaffold — 내 골격 위에 결합부위만 재설계 (본문 6.5)

Level 2 의 "제약 강화"를 가장 강하게 거는 방법이 scaffold 예요. 항체·나노바디뿐 아니라 내가 가진
검증된 단백질 골격을 그대로 두고 결합부위만 다시 설계합니다. scaffold YAML 은 다섯 요소로 이뤄져요.

```yaml
# my_scaffold.yaml
path: my_protein.cif
include:
  - chain: { id: A }
design:                       # 재설계할 영역
  - chain: { id: A, res_index: 26..34,52..59,98..118 }
not_design:                   # 절대 고정 (구조 핵심·기능 잔기)
  - chain: { id: A, res_index: 1..25,35..51,60..97,119.. }
structure_groups:
  - group: { id: A, visibility: 2 }                                   # 골격 구조 유지
  - group: { id: A, visibility: 0, res_index: 26..34,52..59,98..118 } # 결합부위는 자유
design_insertions:            # 결합부위 길이 가변 (loop 연장)
  - insertion: { id: A, res_index: 26, num_residues: 1..5 }
reset_res_index:
  - chain: { id: A }
```

타깃 YAML 에서는 파일로 불러오기만 하면 돼요.

```yaml
entities:
  - file: { path: target.cif, include: [ { chain: { id: B } } ] }
  - file: { path: my_scaffold.yaml }   # 여러 개 나열하면 BoltzGen 이 최적을 선택
```

여러 scaffold 를 동시에 주면 각 골격으로 설계를 시도하고 자동으로 최적을 골라요.
CDR3 길이가 다른 scaffold 를 섞으면(짧은 것=평평한 epitope, 긴 것=깊은 pocket) 다양한 타깃 형태에 대응합니다.
Ch.08(Fab)·Ch.09(나노바디) 실습이 바로 이 전략이에요.

## 9) 모델 자체를 바꾸기 (본문 6.6)

scaffold 로도 부족할 때 손대는 마지막 층이에요. 표준 워크플로우로 안 될 때만 쓰세요.

| 옵션 | 용도 |
|------|------|
| `--design_checkpoints A.ckpt B.ckpt` | 백본 생성 체크포인트 교체(기본 diverse+adherence) |
| `--inverse_fold_checkpoint C.ckpt` | 역접힘 모델 교체(fine-tune 한 가중치 등) |
| `--folding_checkpoint D.ckpt` | 검증(Boltz-2) 체크포인트 교체 |
| `--affinity_checkpoint E.ckpt` | 친화도 예측기 교체 |
| `--step_scale`, `--noise_scale` | 확산 스케줄 고정(탐색 보수성 조절) |

특정 단백질군(막단백질·효소 패밀리 등)에 대해 역접힘 모델을 자체 데이터로 fine-tune 한 뒤
`--inverse_fold_checkpoint` 로 끼워 넣으면 그 도메인에 특화된 서열 설계가 가능해요.

## 10) 교차 검증과 적용 흐름 (본문 6.7·6.8)

여기까지가 BoltzGen 안에서 할 수 있는 전부예요. 신뢰도를 더 올리려면 다른 가정을 쓰는 도구로 다시 걸러야 합니다.
권장 순서는 이렇습니다.

```
BoltzGen(생성·1차 검증)
  → liability / humanness (서열 필터, Ch.08·09)
  → 분자도킹 · MD (물리 검증)
  → 실험
```

- AutoDock Vina — 소분자 결합을 독립적으로 재도킹해 친화도 교차 검증
  (`vina --receptor protein.pdbqt --ligand ligand.pdbqt --out docked.pdbqt`)
- PyMOL — 인터페이스 잔기·수소결합·이황화 거리 시각 검증(`refold_cif` 사용)
- ESM / 서열 분석 — 면역원성·발현성·humanness 예측
- MD 시뮬레이션(GROMACS 등) — 상위 후보의 결합 안정성을 동역학으로 검증

각 단계가 서로 다른 가정으로 거르니, 통과한 후보의 신뢰도가 곱으로 올라가요.
실전 신약 후보 발굴은 이 전부를 한 줄로 꿴 모습이에요.

```
타깃 선정·구조 확보(Ch.02) → 결합부위 전략 수립
   → Level 1 광역 스크리닝(num_designs 10k+)
   → 필터로 좁히기(6.4) → Level 2 집중 재설계 → top 10~50
   → 도킹/MD 교차검증(6.7) → 합성·실험 검증
   → 실험 결과로 필터 기준 개선 → 다음 라운드
```

이 챕터에서 확인한 건 두 가지예요. 부등호 방향 하나로 남는 집합이 정반대가 된다는 것, 그리고
메트릭·조성·크기 필터를 겹칠수록 후보 수는 줄지만 평균 품질은 올라간다는 것. 다음 Part B 부터는
이 선별 기술을 타깃 타입별 실전 설계에 적용해요 — 첫 주자는 고리형 펩타이드입니다.